# Imports

In [1]:
import re
import optuna
import pickle

import numpy as np
import pandas as pd

from sklearn import set_config
from catboost import CatBoostClassifier

from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score
from sklearn.ensemble import StackingClassifier, VotingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict

from feature_engine.selection import DropFeatures

/home/junior/Documentos/GitHub/prediction-f1-pit-stops-kaggle-competition/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Utils

In [2]:
set_config(enable_metadata_routing=True)

In [3]:
def voting_cross_val_predict(probas: list[np.ndarray], weights: np.ndarray[float]) -> np.ndarray[float]:
    weights /= weights.sum()
    return np.sum([proba * weight for proba, weight in zip(probas, weights)], axis=0)

# Loading Dataset

In [4]:
X_train = pd.read_parquet('../data/processed/X_train_stacking.parquet')
y_train = pd.read_parquet('../data/processed/y_train.parquet')

X_test = pd.read_parquet('../data/processed/X_test_stacking.parquet')

In [5]:
X_train.head()

,lgbm_proba,cat_proba,xgb_proba,hist_proba,lg_proba,knn_proba
0,0.802607,0.891870,0.895659,0.883947,0.808497,0.887454
1,0.649405,0.831360,0.829854,0.779541,0.689888,0.519966
2,0.505904,0.851809,0.751522,0.758911,0.558520,0.209855
3,0.001713,0.005910,0.007057,0.007298,0.002275,0.000000
4,0.392205,0.565506,0.489033,0.482967,0.414144,0.297043


In [6]:
X_test.head()

,lgbm_proba,cat_proba,xgb_proba,hist_proba,lg_proba,knn_proba
0,0.004591,0.031986,0.022774,0.020057,0.006419,0.000000
1,0.003032,0.011817,0.014957,0.018887,0.063872,0.117213
2,0.003777,0.011641,0.014307,0.011497,0.004517,0.000000
3,0.163131,0.544040,0.585898,0.485388,0.655870,0.099916
4,0.872172,0.967646,0.959451,0.959356,0.840450,1.000000


## Creating Stacking Database

In [7]:
X_train['voting_proba'] = voting_cross_val_predict(
    [
        X_train['lgbm_proba'],
        X_train['cat_proba'],
        X_train['xgb_proba'],
        X_train['hist_proba']
    ], 
    weights=np.power([0.9484150827066593, 0.9481909576979181, 0.9477770044293032, 0.9485545192992528], 2)
)

In [8]:
X_test['voting_proba'] = voting_cross_val_predict(
    [
        X_test['lgbm_proba'],
        X_test['cat_proba'],
        X_test['xgb_proba'],
        X_test['hist_proba']
    ], 
    weights=np.power([0.9484150827066593, 0.9481909576979181, 0.9477770044293032, 0.9485545192992528], 2)
)

In [9]:
X_train.head()

,lgbm_proba,cat_proba,xgb_proba,hist_proba,lg_proba,knn_proba,voting_proba
0,0.802607,0.891870,0.895659,0.883947,0.808497,0.887454,0.868510
1,0.649405,0.831360,0.829854,0.779541,0.689888,0.519966,0.772514
2,0.505904,0.851809,0.751522,0.758911,0.558520,0.209855,0.717012
3,0.001713,0.005910,0.007057,0.007298,0.002275,0.000000,0.005494
4,0.392205,0.565506,0.489033,0.482967,0.414144,0.297043,0.482416


# Machine Learning

In [10]:
features_to_use = ['lgbm_proba', 'cat_proba', 'xgb_proba', 'hist_proba', 'lg_proba', 'voting_proba']

In [11]:
def objective(trial, X, y):

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    aucs = []

    params = {
        "solver": trial.suggest_categorical("solver", ["saga"]),
        "C": trial.suggest_float("C", 1e-5, 100, log=True),
        "l1_ratio": trial.suggest_float("l1_ratio", 0.0, 1.0),
        "class_weight": trial.suggest_categorical("class_weight", [None, "balanced"]),
        "fit_intercept": trial.suggest_categorical("fit_intercept", [True, False]),
        "tol": trial.suggest_float("tol", 1e-6, 1e-2, log=True),
        "max_iter": trial.suggest_int("max_iter", 1000, 5000)
    }

    for fold, (train_idx, valid_idx) in enumerate(cv.split(X, y)):

        X_train, X_valid = X.iloc[train_idx, :], X.iloc[valid_idx, :]
        y_train, y_valid = y.iloc[train_idx, 0], y.iloc[valid_idx, 0]

        model = LogisticRegression(**params)
        model.fit(X_train, y_train)

        proba = model.predict_proba(X_valid)[:, 1]

        auc = roc_auc_score(y_valid, proba)
        aucs.append(auc)

        trial.report(np.mean(aucs), step=fold)

        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

    return np.mean(aucs)

features_to_use = ['lgbm_proba', 'cat_proba', 'xgb_proba', 'hist_proba', 'lg_proba', 'voting_proba']

study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=42), pruner=optuna.pruners.MedianPruner(n_warmup_steps=2))
study.optimize(lambda trial: objective(trial, X_train[features_to_use], y_train), n_trials=500, n_jobs=-1, show_progress_bar=True)


print("Best trial score:")
print(study.best_trial.value)

print("\nBest params:")
print(study.best_trial.params)

[I 2026-05-18 11:33:41,331] A new study created in memory with name: no-name-d7dab09d-ef7a-46d7-9929-b223e4875dea
Best trial: 9. Best value: 0.5:   0%|          | 1/500 [00:06<52:36,  6.32s/it]

[I 2026-05-18 11:33:47,655] Trial 9 finished with value: 0.5 and parameters: {'solver': 'saga', 'C': 1.5132575896452491e-05, 'l1_ratio': 0.9337103830604095, 'class_weight': 'balanced', 'fit_intercept': False, 'tol': 7.992329255778545e-06, 'max_iter': 2987}. Best is trial 9 with value: 0.5.


Best trial: 8. Best value: 0.799165:   0%|          | 2/500 [00:08<30:49,  3.71s/it]

[I 2026-05-18 11:33:49,540] Trial 8 finished with value: 0.7991645737687293 and parameters: {'solver': 'saga', 'C': 0.012679707484120391, 'l1_ratio': 0.047039597691156954, 'class_weight': None, 'fit_intercept': False, 'tol': 0.008018631956286612, 'max_iter': 1173}. Best is trial 8 with value: 0.7991645737687293.


Best trial: 2. Best value: 0.831978:   1%|          | 3/500 [00:14<39:24,  4.76s/it]

...

[I 2026-05-18 12:15:46,640] Trial 389 finished with value: 0.9496285275871033 and parameters: {'solver': 'saga', 'C': 2.709443970292981, 'l1_ratio': 0.9676053016806354, 'class_weight': None, 'fit_intercept': True, 'tol': 1.0034084829117482e-06, 'max_iter': 2626}. Best is trial 411 with value: 0.9497064502408218.


/home/junior/Documentos/GitHub/prediction-f1-pit-stops-kaggle-competition/.venv/lib/python3.13/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/home/junior/Documentos/GitHub/prediction-f1-pit-stops-kaggle-competition/.venv/lib/python3.13/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
Best trial: 411. Best value: 0.949706: 100%|█████████▉| 499/500 [44:26<01:34, 94.67s/it]

[I 2026-05-18 12:18:07,559] Trial 369 finished with value: 0.9496288062402956 and parameters: {'solver': 'saga', 'C': 1.7618715611112983, 'l1_ratio': 0.9990523645192971, 'class_weight': None, 'fit_intercept': True, 'tol': 1.3931673864445e-06, 'max_iter': 1784}. Best is trial 411 with value: 0.9497064502408218.


/home/junior/Documentos/GitHub/prediction-f1-pit-stops-kaggle-competition/.venv/lib/python3.13/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/home/junior/Documentos/GitHub/prediction-f1-pit-stops-kaggle-competition/.venv/lib/python3.13/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
Best trial: 411. Best value: 0.949706: 100%|██████████| 500/500 [48:13<00:00,  5.79s/it] 

[I 2026-05-18 12:21:54,500] Trial 490 finished with value: 0.949635461850247 and parameters: {'solver': 'saga', 'C': 0.19172016389308552, 'l1_ratio': 0.9983682126462081, 'class_weight': None, 'fit_intercept': True, 'tol': 2.359743729271448e-06, 'max_iter': 1419}. Best is trial 411 with value: 0.9497064502408218.
Best trial score:
0.9497064502408218

Best params:
{'solver': 'saga', 'C': 0.0011427696891455742, 'l1_ratio': 0.9798523189723278, 'class_weight': None, 'fit_intercept': True, 'tol': 2.014092459415771e-06, 'max_iter': 1984}


In [13]:
stacking_lg = LogisticRegression(**study.best_params).fit(X_train, y_train.PitNextLap)

# Submission

In [14]:
submission = pd.read_csv('../data/raw/sample_submission.csv')

In [15]:
submission['PitNextLap'] = stacking_lg.predict_proba(X_test)[:, 1]

In [17]:
submission.to_csv('../data/submission/submission.csv', index=False)